# Poster MNIST images

Generates the 12 PNGs for the poster: 3 test digits × 4 stages (clean, fully blurred, direct one-shot reconstruction, Algorithm 2 iterative).

Outputs land in `poster_images/mnist/` at the project root.

Assumes:
- A trained MNIST checkpoint exists at `checkpoints/mnist.pt`.
- You're running on Colab with the `cs4782-final-project` repo cloned to Drive at `MyDrive/cs4782-final-project`.

## 1. Colab setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
%cd /content/drive/MyDrive/cs4782-final-project
!git pull
!pip install -q -r requirements.txt

os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"

## 2. Imports

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("./code").resolve()))
sys.path.insert(0, str(pathlib.Path(".").resolve()))

import numpy as np
import torch
from PIL import Image

from data.datasets import get_dataset
from model import build_unet
from degradation import Degradation
from sampling import algorithm1_sample, algorithm2_sample, direct_reconstruct

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 3. Load the trained MNIST model + matching degradation

The degradation config below mirrors `DEGRADATION_CONFIGS["mnist"]` in `code/train.py` exactly. If the trainer changes those, change them here too — otherwise sampling will produce garbage.

In [ ]:
# Must match code/train.py:DEGRADATION_CONFIGS["mnist"]
DEG_CFG = dict(
    kernel_size=11,
    num_timesteps=20,
    kernel_std=7.0,
    blur_routine="Constant",
    image_channels=1,
)
T = DEG_CFG["num_timesteps"]

degradation = Degradation(**DEG_CFG).to(device)

model = build_unet("mnist").to(device)

CKPT_PATH = pathlib.Path("checkpoints/mnist.pt")
assert CKPT_PATH.exists(), f"No checkpoint at {CKPT_PATH.resolve()} — train one first via 00_train_mnist.ipynb."

ckpt = torch.load(CKPT_PATH, map_location=device)
# EMA weights produce noticeably better reconstructions than the live model.
model.load_state_dict(ckpt["ema"])
model.eval()

print(f"loaded ckpt from step {ckpt.get('step', '?')} (T={T})")

## 4. Pick test digits and generate the 12 images

Three indices into the MNIST **test** split. Edit if you want different digits — pin them so the poster is reproducible.

In [ ]:
TEST_INDICES = [0, 17, 42]   # one easy, one mid, one curly — adjust if a different mix looks better

OUT_DIR = pathlib.Path("poster_images/mnist")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_png(t, path):
    """Save a tensor as a native-res 8-bit grayscale PNG with no chrome."""
    a = t.detach().cpu().squeeze().numpy()
    a = np.clip(a, 0.0, 1.0)
    a = (a * 255.0).round().astype(np.uint8)
    Image.fromarray(a, mode="L").save(path)

mnist_test = get_dataset("mnist", "test")
print(f"MNIST test set size: {len(mnist_test)}")

with torch.no_grad():
    for idx in TEST_INDICES:
        x0_img, _ = mnist_test[idx]
        x0 = x0_img.unsqueeze(0).to(device)            # (1,1,28,28)

        xT     = degradation.forward_to_step(x0, T)
        direct = direct_reconstruct(model, xT, T)
        alg2   = algorithm2_sample(model, degradation, xT, T)

        save_png(x0,     OUT_DIR / f"mnist_{idx}_x0.png")
        save_png(xT,     OUT_DIR / f"mnist_{idx}_xT.png")
        save_png(direct, OUT_DIR / f"mnist_{idx}_direct.png")
        save_png(alg2,   OUT_DIR / f"mnist_{idx}_alg2.png")
        print(f"  idx={idx}: saved x0, xT, direct, alg2")

print(f"\nall 12 PNGs in {OUT_DIR.resolve()}")

## 5. Bonus: Algorithm 1 stability comparison + half-blur frame

For the poster's stability callout (Alg 1 drifts, Alg 2 stable). Just one example digit is enough.

In [ ]:
BONUS_IDX = TEST_INDICES[0]

with torch.no_grad():
    x0_img, _ = mnist_test[BONUS_IDX]
    x0 = x0_img.unsqueeze(0).to(device)

    xHalf = degradation.forward_to_step(x0, T // 2)
    xT    = degradation.forward_to_step(x0, T)
    alg1  = algorithm1_sample(model, degradation, xT, T)

    save_png(xHalf, OUT_DIR / f"mnist_{BONUS_IDX}_xHalf.png")
    save_png(alg1,  OUT_DIR / f"mnist_{BONUS_IDX}_alg1.png")

print(f"bonus images saved for idx={BONUS_IDX}")

## 6. Sanity check — display inline

Verify the four-stage progression looks right before you ship the PNGs to the poster.

In [ ]:
import matplotlib.pyplot as plt

stages = ["x0", "xT", "direct", "alg2"]
fig, axes = plt.subplots(len(TEST_INDICES), len(stages), figsize=(8, 2 * len(TEST_INDICES)))

for r, idx in enumerate(TEST_INDICES):
    for c, stage in enumerate(stages):
        img = Image.open(OUT_DIR / f"mnist_{idx}_{stage}.png")
        ax = axes[r, c] if len(TEST_INDICES) > 1 else axes[c]
        ax.imshow(img, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        ax.set_title(f"idx={idx} · {stage}", fontsize=9)
        ax.axis("off")

plt.tight_layout()
plt.show()